# Coverage of a Mars geological feature

Pick a feature, confirm, and see what every instrument has observed of it.

- **Per-observation coverage** shows what a single observation covered, as a share of the
  feature's bounding box, at the time it was taken.
- **Cumulative coverage** shows how much of the feature each instrument has reached in
  total, as its observations accumulate.

Run every cell in order. Re-selecting a feature and confirming again only needs the
plotting cells re-run.

Requires the coverage stage to have run: `uv run python scripts/compute_coverage.py`.

## Setup

In [ ]:
"""Locate the repository and load the pipeline's own layout rules."""

import json
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display
from matplotlib.ticker import PercentFormatter


def repository_root() -> Path:
    """Find the repository root from wherever the kernel happens to have started.

    Returns:
        The first directory at or above the working directory holding `data`.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("no `data` directory at or above the working directory")


ROOT = repository_root()
sys.path.insert(0, str(ROOT / "src"))

from analysis import configs  # noqa: E402
from download.storage.layout import slugify  # noqa: E402

COVERAGE_ROOT = ROOT / configs.COVERAGE_ROOT
CATALOG = ROOT / configs.DATA_ROOT / "_catalog"

print(f"repository: {ROOT}")
print(f"coverage artifacts: {COVERAGE_ROOT}")

In [ ]:
"""Read the feature catalogue and note which features have coverage computed locally."""


def read_jsonl(path: Path) -> list[dict]:
    """Read a JSON lines file.

    Args:
        path: The file to read.

    Returns:
        One dictionary per line.
    """
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def feature_directory(feature_class: str, name: str) -> Path:
    """Return where one feature's coverage artifacts live.

    Args:
        feature_class: The feature class, such as Crater.
        name: The feature name as ODE spells it.

    Returns:
        The artifact directory, which need not exist.
    """
    return COVERAGE_ROOT / slugify(feature_class) / slugify(name)


def has_coverage(feature_class: str, name: str) -> bool:
    """Report whether a feature has computed coverage on disk.

    Args:
        feature_class: The feature class, such as Crater.
        name: The feature name as ODE spells it.

    Returns:
        True when at least one instrument set has been computed for it.
    """
    return any(feature_directory(feature_class, name).glob("*.events.parquet"))


FEATURES: dict[str, list[str]] = {}
for entry in read_jsonl(CATALOG / "features.jsonl"):
    FEATURES.setdefault(entry["feature_class"], []).append(entry["name"])
for names in FEATURES.values():
    names.sort()

AVAILABLE = {
    (feature_class, name)
    for feature_class, names in FEATURES.items()
    for name in names
    if has_coverage(feature_class, name)
}

catalogued = sum(len(names) for names in FEATURES.values())
print(f"{catalogued} catalogued features in {len(FEATURES)} classes")
print(f"{len(AVAILABLE)} with coverage computed locally")

## Select a feature

Feature types and names that have no local data are marked. Confirming one of those
shows the grey panel below instead of plots, so it is always clear whether a missing
line means "not observed" or "not downloaded".

In [ ]:
"""Build the feature selector and hold the confirmed choice for the plotting cells."""

_GREY = "#8a8a8a"


def unavailable_panel(message: str) -> widgets.HTML:
    """Build the grey panel shown in place of a plot when there is no data.

    Args:
        message: The line explaining what is missing.

    Returns:
        The rendered panel.
    """
    return widgets.HTML(
        f"""<div style="
            background: repeating-linear-gradient(45deg,
                #ebebeb, #ebebeb 10px, #e0e0e0 10px, #e0e0e0 20px);
            border: 1px solid #c4c4c4; border-radius: 6px; color: {_GREY};
            font-family: sans-serif; padding: 28px; text-align: center;">
          <div style="font-size: 15px; font-weight: 600;">No local data</div>
          <div style="font-size: 13px; margin-top: 6px;">{message}</div>
        </div>"""
    )


def label_for(feature_class: str, name: str) -> str:
    """Return a dropdown label marking whether a feature has local data.

    Args:
        feature_class: The feature class, such as Crater.
        name: The feature name as ODE spells it.

    Returns:
        The name, suffixed when nothing has been computed for it.
    """
    return name if (feature_class, name) in AVAILABLE else f"{name}  (no data)"


class_dropdown = widgets.Dropdown(
    options=sorted(FEATURES),
    description="Type:",
    value="Crater",
    layout=widgets.Layout(width="340px"),
)
name_dropdown = widgets.Dropdown(
    description="Name:", layout=widgets.Layout(width="340px")
)
confirm_button = widgets.Button(
    description="Confirm", button_style="primary", icon="check"
)
status = widgets.Output()

selection: dict[str, str] = {}


def _refresh_names(_change=None) -> None:
    """Repopulate the name dropdown for the selected feature class.

    Args:
        _change: The widget change event, ignored.

    Returns:
        None.
    """
    feature_class = class_dropdown.value
    name_dropdown.options = [
        (label_for(feature_class, name), name) for name in FEATURES[feature_class]
    ]


def _confirm(_button=None) -> None:
    """Record the confirmed feature and report whether it has data.

    Args:
        _button: The button that was clicked, ignored.

    Returns:
        None.
    """
    feature_class, name = class_dropdown.value, name_dropdown.value
    selection.clear()
    selection.update(feature_class=feature_class, name=name)
    status.clear_output()
    with status:
        if (feature_class, name) in AVAILABLE:
            print(f"Selected {feature_class} / {name}. Run the cells below.")
        else:
            missing = f"{feature_class} / {name}"
            display(
                unavailable_panel(
                    f"Nothing has been downloaded or computed for {missing}."
                )
            )


class_dropdown.observe(_refresh_names, names="value")
confirm_button.on_click(_confirm)
_refresh_names()

controls = widgets.HBox([class_dropdown, name_dropdown, confirm_button])
display(widgets.VBox([controls, status]))

## Load the confirmed feature's coverage

In [ ]:
"""Read the confirmed feature's event rows, one table per instrument set."""


def set_label(row: dict) -> str:
    """Return a short readable name for an instrument set.

    Args:
        row: Any event or summary row carrying the set's identifiers.

    Returns:
        The instrument and product type, such as "CTX EDR".
    """
    return f"{row['iid']} {row['pt']}"


def load_sets(feature_class: str, name: str) -> dict[str, dict]:
    """Read every computed instrument set for one feature.

    Args:
        feature_class: The feature class, such as Crater.
        name: The feature name as ODE spells it.

    Returns:
        Each set's events, summary, and gridded flag, keyed by its label and
        ordered by how much of the feature the set covers.
    """
    loaded = {}
    directory = feature_directory(feature_class, name)
    for events_path in sorted(directory.glob("*.events.parquet")):
        summary_path = events_path.with_name(
            events_path.name.replace(".events.parquet", ".summary.parquet")
        )
        if not summary_path.exists():
            continue
        summary = pq.read_table(summary_path).to_pylist()[0]
        loaded[set_label(summary)] = {
            "events": pq.read_table(events_path).to_pylist(),
            "summary": summary,
            "gridded": summary["gridded"],
        }
    ranked = sorted(
        loaded.items(), key=lambda item: -item[1]["summary"]["covered_frac"]
    )
    return dict(ranked)


if not selection:
    display(unavailable_panel("Confirm a feature above first."))
    SETS, AREA_KM2, COLOURS, TITLE = {}, 0.0, {}, ""
else:
    TITLE = f"{selection['feature_class']} / {selection['name']}"
    SETS = load_sets(selection["feature_class"], selection["name"])
    AREA_KM2 = next(iter(SETS.values()))["summary"]["feature_area_km2"] if SETS else 0.0
    COLOURS = {
        label: ("#b0b0b0" if data["gridded"] else colour)
        for (label, data), colour in zip(
            SETS.items(), plt.cm.tab10.colors * 3, strict=False
        )
    }
    print(f"{TITLE}: {AREA_KM2:,.1f} km2 bounding box")
    for label, data in SETS.items():
        note = "  (whole-planet basemap)" if data["gridded"] else ""
        print(f"  {label:16s} {data['summary']['n_obs']:6,d} observations{note}")

## Per-observation coverage

One point per observation, at the time it was taken. The height is the share of the
feature's bounding box that single observation covered, so a tall point is a wide swath
and a low point is a narrow one. Whole-planet basemaps are drawn in grey: they cover
everything by construction and say nothing about targeted observing.

In [ ]:
"""Plot what each single observation covered, one panel per instrument."""


def plot_per_observation(sets: dict[str, dict], area_km2: float, title: str) -> None:
    """Draw one stacked panel per instrument set, sharing both axes.

    Args:
        sets: The loaded instrument sets, keyed by label.
        area_km2: The feature's bounding box area, used to scale coverage.
        title: The figure title.

    Returns:
        None.
    """
    figure, axes = plt.subplots(
        len(sets), 1, figsize=(11, 1.9 * len(sets)), sharex=True, sharey=True
    )
    for axis, (label, data) in zip(np.atleast_1d(axes), sets.items(), strict=True):
        events = data["events"]
        axis.scatter(
            [event["t_start"] for event in events],
            [event["own_km2"] / area_km2 for event in events],
            s=12,
            alpha=0.65,
            color=COLOURS[label],
            edgecolors="none",
        )
        axis.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
        axis.set_ylim(-0.05, 1.05)
        axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        axis.grid(axis="y", alpha=0.25, linewidth=0.5)
        axis.spines[["top", "right"]].set_visible(False)
    axes_list = np.atleast_1d(axes)
    axes_list[0].set_title(title, fontsize=12, loc="left")
    axes_list[-1].set_xlabel("Observation start time")
    figure.supylabel("Share of the feature covered by one observation", fontsize=10)
    figure.tight_layout()
    plt.show()


if not SETS:
    display(unavailable_panel("No instrument set has been computed for this feature."))
else:
    plot_per_observation(
        SETS,
        AREA_KM2,
        f"{TITLE}  -  coverage per observation",
    )

## Cumulative coverage

The left panel is the running union of everything an instrument has observed, so it only
ever rises and it flattens once an instrument stops finding new ground. The right panel
is where each instrument ended up.

In [ ]:
"""Plot how much of the feature each instrument reaches over time, and in total."""


def plot_cumulative(sets: dict[str, dict], title: str) -> None:
    """Draw the running coverage per instrument beside its final total.

    Args:
        sets: The loaded instrument sets, keyed by label.
        title: The figure title.

    Returns:
        None.
    """
    figure, (curve, bars) = plt.subplots(
        1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [3, 1]}
    )
    for label, data in sets.items():
        events = data["events"]
        curve.plot(
            [event["t_start"] for event in events],
            [event["cum_frac"] for event in events],
            drawstyle="steps-post",
            linewidth=1.8,
            linestyle="--" if data["gridded"] else "-",
            color=COLOURS[label],
            label=f"{label}  ({data['summary']['covered_frac']:.1%})",
        )
    curve.set_title(title, fontsize=12, loc="left")
    curve.set_xlabel("Observation start time")
    curve.set_ylabel("Share of the feature covered so far")
    curve.set_ylim(0, 1.05)
    curve.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    curve.grid(alpha=0.25, linewidth=0.5)
    curve.spines[["top", "right"]].set_visible(False)
    curve.legend(fontsize=9, loc="lower right", frameon=False)

    labels = list(sets)[::-1]
    bars.barh(
        labels,
        [sets[label]["summary"]["covered_frac"] for label in labels],
        color=[COLOURS[label] for label in labels],
    )
    bars.set_title("Total covered", fontsize=11, loc="left")
    bars.set_xlim(0, 1.05)
    bars.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    bars.tick_params(labelsize=9)
    bars.grid(axis="x", alpha=0.25, linewidth=0.5)
    bars.spines[["top", "right"]].set_visible(False)
    figure.tight_layout()
    plt.show()


if not SETS:
    display(unavailable_panel("No instrument set has been computed for this feature."))
else:
    plot_cumulative(
        SETS,
        f"{TITLE}  -  cumulative coverage",
    )